# Pipeline completo de diez etapas

Este notebook usa el punto de entrada final, con órdenes sintéticas explícitas y datos medidos por separado. Los notebooks 00–04 conservan el desarrollo histórico.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.complete_pipeline import CompletePipeline
from src.data_source import synthetic_events
from threadpoolctl import threadpool_limits


In [2]:
with threadpool_limits(1):
    pipeline = CompletePipeline(ROOT / "data/processed", expected_items=5000)
    pipeline.run(synthetic_events(5000))
summary = pipeline.summary()
summary["metrics"], summary["stage_counts"]

({'received': 5000,
  'valid': 5000,
  'invalid': 0,
  'duplicates': 51,
  'output': 4949,
  'bloom_positives': 10,
  'bloom_false_positives': 10,
  'exact_checks': 10,
  'cms_candidates': 40,
  'cms_false_candidates': 0,
  'inferred_purchases': 0,
  'synthetic_events': 4949,
  'snapshots': 1,
  'sales_cents': 44713200},
 {'0': 5000,
  '1': 5000,
  '2': 4949,
  '3': 4949,
  '4': 200,
  '5': 200,
  '6': 4949,
  '7': 200,
  '8': 2475,
  '9': 200})

## Índices y recomendaciones
El filtro de precios consulta páginas del B+Tree; las recomendaciones consumen IVF, LSH, CSR y grafo.

In [3]:
pid = next(iter(pipeline.catalog))
assert all(summary["stage_counts"][str(i)] > 0 for i in range(10))
pipeline.bptree.range_search(1000, 10000)[:10], pipeline.recommend(pid)

([1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
 {'product': {'product_id': 1,
   'category': 'electronics.group0',
   'brand': 'marca0',
   'price': 11.7,
   'views': 5,
   'carts': 4,
   'purchases': 36,
   'stock': 64,
   'stock_source': 'simulated_initial_stock_100'},
  'recommendations': [{'product_id': 3,
    'category': 'electronics.group0',
    'brand': 'marca0',
    'price': 15.1,
    'views': 8,
    'carts': 6,
    'purchases': 10,
    'stock': 90,
    'stock_source': 'simulated_initial_stock_100',
    'score': 2.8392133245969644,
    'reasons': ['Jaccard', 'co-compra'],
    'pagerank': 0.004015801150372771,
    'community': 0},
   {'product_id': 2,
    'category': 'electronics.group0',
    'brand': 'marca0',
    'price': 13.4,
    'views': 10,
    'carts': 4,
    'purchases': 10,
    'stock': 90,
    'stock_source': 'simulated_initial_stock_100',
    'score': 2.8386798647899116,
    'reasons': ['Jaccard', 'co-compra'],
    'pagerank': 0.004015801150372771,
    'community': 0},
   {'produc

In [4]:
pipeline.graph_analysis["comparison"]

[{'algorithm': 'louvain',
  'mean_time_s': 0.0037405333326508603,
  'std_time_s': 0.00031655116546483203,
  'modularity': 0.8002483409690407,
  'community_count': 6,
  'community_sizes': [40, 40, 40, 40, 21, 19],
  'stability_ari': 0.951487656263545,
  'stability_runs': 3},
 {'algorithm': 'label_propagation',
  'mean_time_s': 0.002347366651520133,
  'std_time_s': 0.00011460517476308794,
  'modularity': 0.6755584062196306,
  'community_count': 26,
  'community_sizes': [16,
   15,
   14,
   12,
   11,
   10,
   9,
   9,
   9,
   9,
   9,
   6,
   6,
   6,
   6,
   6,
   6,
   5,
   5,
   5,
   5,
   5,
   4,
   4,
   4,
   4],
  'stability_ari': 0.5500380338426473,
  'stability_runs': 3}]

In [5]:
summary["alerts"], summary["trace"][0]

([{'window': '2026-10-30T00:00:00+00:00',
   'user_id': 999999,
   'event_id': 48,
   'estimated': 10,
   'exact': 10}],
 {'event_id': 17,
  'product_id': 194,
  'priority': 0,
  'type': 'purchase',
  'bloom_positive': False,
  'alert': None,
  'completed_stages': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
  'snapshot': {'price_index_contains_product': True,
   'ann_neighbors': [{'product_id': 127, 'distance': 0.5317851904993041},
    {'product_id': 179, 'distance': 0.5707340511478068},
    {'product_id': 157, 'distance': 0.6251339530448626}],
   'community': 5,
   'graph_member': True,
   'graph_action': 'purchase_updates_order'}})

## Resultados reales y límites
Los JSON proceden de ejecuciones REES46 independientes. N es eventos de origen; los tamaños de índices y grafo se informan por separado.

In [6]:
import pandas as pd
records = []
for path in sorted((ROOT / "results").glob("complete_rees46_*/run_metadata.json")):
    records.append(json.loads(path.read_text(encoding="utf-8")))
pd.DataFrame(records).sort_values("requested_events")

,source,requested_events,received_events,rss_initial_mib,rss_final_mib,rss_peak_mib,rss_peak_increment_mib,rss_sample_interval_s,python,platform,cpu,ram_gib,seed,native_threads,elapsed_s,products,graph_nodes,graph_edges
0,rees46,1000,1000,172.363281,179.187500,179.187500,6.824219,0.02,3.12.10,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 186 Stepping 2, Genuine...",15.668816,42,1,0.184531,534,10,1
1,rees46,10000,10000,172.535156,199.167969,199.167969,26.632812,0.02,3.12.10,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 186 Stepping 2, Genuine...",15.668816,42,1,1.176495,4148,96,6
2,rees46,100000,100000,172.308594,268.093750,268.652344,96.343750,0.02,3.12.10,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 186 Stepping 2, Genuine...",15.668816,42,1,14.597456,20621,843,159
3,rees46,1000000,1000000,172.570312,461.457031,481.355469,308.785156,0.02,3.12.10,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 186 Stepping 2, Genuine...",15.668816,42,1,85.094655,63322,4549,1420
4,rees46,10000000,10000000,172.375000,817.941406,1064.261719,891.886719,0.02,3.12.10,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 186 Stepping 2, Genuine...",15.668816,42,1,1133.126976,121903,19215,12607


In [7]:
pipeline.close()